In [19]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import PoissonRegressor
import pandas as pd
from sklearn.metrics import mean_poisson_deviance, mean_squared_error
from sklearn.model_selection import cross_val_score, KFold
import matplotlib.pyplot as plt
import numpy as np
import json

In [2]:
raw_df = pd.read_csv("telematics_syn-032021.csv")
print(f"Length of raw_df: {len(raw_df)}")
# print(f"Minimum Insured.age: {min(raw_df['Insured.age'])}")
# Drop rows where Insured.age - Years.noclaims < 16, as these are likely data entry errors
clean_df = raw_df[raw_df["Insured.age"] - raw_df["Years.noclaims"] >= 16].copy()
print(f"Length of clean_df after dropping rows with unexpected years of no claims: {len(clean_df)}")

# convert the "Territory" column to a categorical type for VIF calculation
clean_df["Territory"] = clean_df["Territory"].astype("category")

X = clean_df.drop(columns=['NB_Claim', 'AMT_Claim', 'Duration'])
# Remove redundant columns that are highly correlated with other features
X = X.drop(columns=['Pct.drive.sun', 'Pct.drive.wkday', 'Pct.drive.wkend'])
y_num = clean_df['NB_Claim']
y_amt = clean_df['AMT_Claim']
exposure = clean_df['Duration']/366  # Convert exposure to years


print(f"Clean_df shape: {clean_df.shape}")
print(f"X shape: {X.shape}")

Length of raw_df: 100000
Length of clean_df after dropping rows with unexpected years of no claims: 98541
Clean_df shape: (98541, 52)
X shape: (98541, 46)


In [7]:
traditional_features = X[['Insured.age', 'Insured.sex', 'Car.age', 'Marital',
       'Car.use', 'Credit.score', 'Region', 'Annual.miles.drive',
       'Years.noclaims', 'Territory']]
telematics_features = X.drop(columns=traditional_features)

In [8]:
X_train, X_test, y_train, y_test, exposure_train, exposure_test = train_test_split(
    telematics_features, y_num, exposure, test_size=0.2, random_state=42, stratify=y_num
)

In [9]:
def poisson_cv_scores(X, y_num, exposure, n_splits=5, alpha=0.0, random_state=42):
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=random_state)
    d2_scores = []
    deviance_scores = []
    for train_idx, val_idx in kf.split(X):
        X_tr, X_val = X.iloc[train_idx], X.iloc[val_idx]
        y_tr, y_val = y_num.iloc[train_idx], y_num.iloc[val_idx]
        exp_tr, exp_val = exposure.iloc[train_idx], exposure.iloc[val_idx]

        scaler = StandardScaler()
        X_tr = scaler.fit_transform(X_tr)
        X_val = scaler.transform(X_val)

        model = PoissonRegressor(alpha=alpha)
        model.fit(X_tr, y_tr / exp_tr, sample_weight=exp_tr)

        y_val_pred = model.predict(X_val)
        d2_scores.append(model.score(X_val, y_val / exp_val, sample_weight=exp_val))
        deviance_scores.append(mean_poisson_deviance(y_val / exp_val, y_val_pred, sample_weight=exp_val))
    return np.array(d2_scores), np.array(deviance_scores)

In [20]:
groups = {
    "accel": [col for col in X.columns if col.startswith("Accel.")],
    "brake": [col for col in X.columns if col.startswith("Brake.")],
    "left":  [col for col in X.columns if col.startswith("Left.turn")],
    "right": [col for col in X.columns if col.startswith("Right.turn")],
    "hrs":   [col for col in X.columns if col.startswith("Pct.drive") and col.endswith("hrs")]
}

# starting choice as lowest threshold
chosen = {name: cols[0] for name, cols in groups.items()}

all_group_cols = [col for cols in groups.values() for col in cols]
base_cols = [col for col in X_train.columns if col not in all_group_cols]

groups
# test one group at a time
for name, cols in groups.items():
    results = []
    for candidate in cols:
        trial = {**chosen, name: candidate}           
        features = base_cols + list(trial.values())
        d2, dev = poisson_cv_scores(X_train[features], y_train, exposure_train)
        print(dev)
        results.append((candidate, dev.mean(), dev.std()))
    for candidate, mean_dev, std_dev in results:
        print(f"  {candidate}: {mean_dev:.4f} (+/- {std_dev:.4f})")
    # update the chosen column for this group to the one with the lowest mean deviance
    chosen[name] = min(results, key=lambda x: x[1])[0]


drop_columns = [col for name, cols in groups.items() for col in cols if col != chosen[name]]
json.dump(drop_columns, open("drop_columns.json", "w"), indent=2)

[0.28735396 0.29063216 0.28527139 0.28440581 0.27993625]
[0.28784108 0.2907064  0.28508681 0.28454692 0.28026142]
[0.28783639 0.29070054 0.28506531 0.2846028  0.28027929]
[0.28782119 0.29069883 0.28506121 0.28460482 0.28026527]
[0.28779636 0.29069185 0.28507631 0.28462646 0.28024968]
[0.28779284 0.29068507 0.28506309 0.28462233 0.28025432]
  Accel.06miles: 0.2855 (+/- 0.0035)
  Accel.08miles: 0.2857 (+/- 0.0035)
  Accel.09miles: 0.2857 (+/- 0.0035)
  Accel.11miles: 0.2857 (+/- 0.0035)
  Accel.12miles: 0.2857 (+/- 0.0035)
  Accel.14miles: 0.2857 (+/- 0.0035)
[0.28735396 0.29063216 0.28527139 0.28440581 0.27993625]
[0.28939874 0.29155427 0.28738955 0.28496614 0.28133714]
[0.2895554  0.29208539 0.28800135 0.28542933 0.2816891 ]
[0.28950833 0.29213712 0.28806407 0.28554578 0.28168383]
[0.28944453 0.29212948 0.28805565 0.28555729 0.28166133]
[0.28941379 0.29209562 0.28807576 0.28555612 0.28164817]
  Brake.06miles: 0.2855 (+/- 0.0035)
  Brake.08miles: 0.2869 (+/- 0.0035)
  Brake.09miles: 0.2